# Evaluation Pipeline

We will in this notebook setup the evaluation pipeline. We generated questions in ./generate_factuality_questions.ipynb that contain entries

- id
- question
- correct answer

Where we can sample 4 entries from annotated_dataset along with the incorrect entry, where the question pertains to. We will then evaluate the answer to the question by comparing it to the answer. We will use LLM-as-a-Judge and give it

- correct answer
- answer from LLM

and make it respond with something along the lines of

```json
{
    accuracy: 90%
}
```

In [2]:
import os

import pandas as pd

path_to_questions_answers = "data/factuality_questions_answers.pkl"

if not os.path.exists(path_to_questions_answers):
    raise FileNotFoundError(
        "File does not exist. See ./generate_factuality_questions.ipynb to generate the file."
    )

df_questions = pd.read_pickle(path_to_questions_answers)
df_questions

,topic_id,question,answer
0,ethiopia_challenges__btithihhtt,Which group overthrew the Derg in 1991?,Ethiopian People's Revolutionary Democratic Front
1,ethiopia_challenges__btithihhtt,What did Prime Minister Abiy Ahmed pledge to d...,Reform the country
2,ethiopia_challenges__btithihhtt,What group did Abiy Ahmed begin peace talks wi...,Tigray People's Liberation Front
3,topic_260__mtaftfttit,What percentage of female characters in top 20...,36.3 percent
4,topic_260__mtaftfttit,What does the text say about female characters...,Only 36.3 percent had speaking roles
5,topic_260__mtaftfttit,"According to the Geena Davis Institute, what p...",Not explicitly confirmed
6,topic_493__isitgsgshgsg,When did Indira Gandhi begin her political car...,In 1955
7,topic_493__isitgsgshgsg,Who was Indira Gandhi's father?,Jawaharlal Nehru
8,topic_493__isitgsgshgsg,What happened to Indira Gandhi in 1984?,She was assassinated
9,topic_131__gtttgtiigt,When was tea introduced to North America?,In the 17th century


In [3]:
def sample_entries(df: pd.DataFrame, id: str, n: int = 5) -> pd.DataFrame:
    """Sample n entries from the dataframe with a specific id.

    Args:
        df (pd.DataFrame): The dataframe to sample from.
        id (str): The id to filter by.
        n (int, optional): The number of samples to return. Defaults to 5. Must atleast be 1, as one entry with the given id is always included.

    Returns:
        pd.DataFrame: The sampled dataframe.
    """
    samples = df[df["id"] != id].sample(n - 1, random_state=22)
    samples = pd.concat([df[df["id"] == id], samples])
    return samples

In [4]:
from utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = "smoldoc__en_sw"  # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset  # TODO we do not actually need the annotated version here, just the basic version

📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data/smoldoc-factuality-ratings.json


Dataset({
    features: ['id', 'sl', 'tl', 'srcs', 'trgs', 'factuality', 'is_src_orig', 'annotator_1_label', 'annotator_1_notes', 'annotator_2_label', 'annotator_2_notes', 'annotator_3_label', 'annotator_3_notes'],
    num_rows: 584
})

In [5]:
annotated_df = pd.DataFrame(annotated_dataset)
annotated_df.head()

,id,sl,tl,srcs,trgs,factuality,is_src_orig,annotator_1_label,annotator_1_notes,annotator_2_label,annotator_2_notes,annotator_3_label,annotator_3_notes
0,topic_183__dtlihtiibiiiii,en,sw,"[Dude, you won't believe this., There's this e...","[Mwenzangu, huwezi kuamini hili., Pana mhandis...",ok,True,N/A - Not Applicable,No factual claims are made. It's a fictional s...,N/A - Not Applicable,This is a fictious story about someone's caffe...,N/A - Not Applicable,No factual claims are made. It's about a perso...
1,topic_16__ittbtiyrnwyriytr,en,sw,"[""I stood before the sculpture, my heart fille...","[""Nilisimama mbele ya mchongo huo, moyo wangu ...",ok,True,N/A - Not Applicable,No factual claims are made. It's a fictional s...,N/A - Not Applicable,It is a fictional narrative about an art discu...,N/A - Not Applicable,No factual claims are made. This is a conversi...
2,topic_230__tttatttt,en,sw,"[The old man's wake was a raucous affair, as b...",[Matanga ya mzee huyo yalikuwa na mchakamchaka...,ok,True,N/A - Not Applicable,No factual claims are made. It's a fictional s...,N/A - Not Applicable,It is a fictional narrative about a wake. No f...,N/A - Not Applicable,No factual claims are made. The paragraph desc...
3,topic_232__fydeggp,en,sw,"[Face and Body Care, Your skin is your largest...","[Utunzaji wa Uso na Mwili, Ngozi yako ndicho k...",ok,True,No Issues,There are some face and body skincare tips her...,No Issues,The claims about skincare are all true and fac...,No Issues,It is true that the skin is the largest organ ...
4,ethiopia_challenges__btithihhtt,en,sw,"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...",has_errors,True,Minor Issue(s),"This paragraph contains minor issue: the ""peac...",No Issues,This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...


In [13]:
from tqdm.notebook import tqdm

from llm_chat import CachedLLMChat, LLMChat, OllamaChatter


verbose = False

chatter = OllamaChatter(model_name="gemma3:4b")
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="data/evaluation_cache.pkl")

answers: list[dict[str, str]] = []

for id, question, answer in tqdm(
    df_questions.itertuples(index=False, name=None),
    total=len(df_questions),
    desc="Evaluating factuality questions",
):
    samples = sample_entries(annotated_df, id, n=5)

    if verbose:
        print(f"Question originates from document with ID: {id}")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        print(f"Using document samples with IDs: {' '.join(samples['id'])}")
        print("\n")

    chat.add_message("system", "You are an expert translator from English to Swahili. ")
    for idx, row in samples.iterrows():
        srcs = " ".join(row["srcs"])
        trgs = " ".join(row["trgs"])
        chat.add_message(
            "user", f"Translate this document from English to Swahili:\n\n{srcs}"
        )
        chat.add_message("assistant", trgs)
    chat.add_message(
        "system",
        "You are now an expert question answerer. Provide succinct answers in English to the questions asked. Do not answer in other languages.",
    )
    response, _ = chat.chat(question)
    print(f"Q: {question}\nA: {response}\n(Expected: {answer})\n{'-' * 80}\n")
    chat.reset()

    # collect answer and correct answer for evaluation later
    answers.append(
        {
            "id": id,
            "question": question,
            "expected answer": answer,
            "answer": response,
        }
    )


Evaluating factuality questions:   0%|          | 0/15 [00:00<?, ?it/s]

Q: Which group overthrew the Derg in 1991?
A: The Ethiopian People’s Revolutionary Democratic Front (EPRDF).
(Expected: Ethiopian People's Revolutionary Democratic Front)
--------------------------------------------------------------------------------

Q: What did Prime Minister Abiy Ahmed pledge to do in 2018?
A: He pledged to reform the country.
(Expected: Reform the country)
--------------------------------------------------------------------------------

Q: What group did Abiy Ahmed begin peace talks with in 2018?
A: The Tigray People’s Liberation Front (TPLF).
(Expected: Tigray People's Liberation Front)
--------------------------------------------------------------------------------

Q: What percentage of female characters in top 2017 films had speaking roles?
A: 30%.
(Expected: 36.3 percent)
--------------------------------------------------------------------------------

Q: What does the text say about female characters in top 2017 films?
A: The text states that only 30% of fem

In [14]:
answers_df = pd.DataFrame(answers)
answers_df

,id,question,expected answer,answer
0,ethiopia_challenges__btithihhtt,Which group overthrew the Derg in 1991?,Ethiopian People's Revolutionary Democratic Front,The Ethiopian People’s Revolutionary Democrati...
1,ethiopia_challenges__btithihhtt,What did Prime Minister Abiy Ahmed pledge to d...,Reform the country,He pledged to reform the country.
2,ethiopia_challenges__btithihhtt,What group did Abiy Ahmed begin peace talks wi...,Tigray People's Liberation Front,The Tigray People’s Liberation Front (TPLF).
3,topic_260__mtaftfttit,What percentage of female characters in top 20...,36.3 percent,30%.
4,topic_260__mtaftfttit,What does the text say about female characters...,Only 36.3 percent had speaking roles,The text states that only 30% of female charac...
5,topic_260__mtaftfttit,"According to the Geena Davis Institute, what p...",Not explicitly confirmed,12%.
6,topic_493__isitgsgshgsg,When did Indira Gandhi begin her political car...,In 1955,1955
7,topic_493__isitgsgshgsg,Who was Indira Gandhi's father?,Jawaharlal Nehru,Jawaharlal Nehru.
8,topic_493__isitgsgshgsg,What happened to Indira Gandhi in 1984?,She was assassinated,She was assassinated in 1984.
9,topic_131__gtttgtiigt,When was tea introduced to North America?,In the 17th century,Tea was introduced to North America in the 18t...


## Evaluating answers

In [17]:
system_prompt = """\
You are an expert evaluator that will score answers based on their factual accuracy.

The answer you provide should be an integer from 1 to 5, where:
1 - The answer is completely incorrect or irrelevant.
2 - The answer has significant inaccuracies or omissions.
3 - The answer is partially correct but lacks important details.
4 - The answer is mostly correct with minor inaccuracies.
5 - The answer is completely correct and comprehensive.

Remember: Your answer should only be an integer between 1 and 5, and nothing else!
"""

scores: list[int] = []

for id, q, expected, a in answers_df.itertuples(index=False, name=None):
    chat.add_message("system", system_prompt)
    response, _ = chat.chat(f"Question: {q}\nExpected answer: {expected}\nAnswer given: {a}\n\nPlease provide a score from 1 to 5.")
    print(f"Question: {q}")
    print(f"Expected answer: {expected}")
    print(f"Answer given: {a}")
    print(f"ID: {id}\nScore: {response}\n{'-'*80}\n")
    scores.append(int(response))
    chat.reset()

average_score = sum(scores) / len(scores)
print(f"Average factuality score: {average_score:.2f} out of 5")

Question: Which group overthrew the Derg in 1991?
Expected answer: Ethiopian People's Revolutionary Democratic Front
Answer given: The Ethiopian People’s Revolutionary Democratic Front (EPRDF).
ID: ethiopia_challenges__btithihhtt
Score: 5

--------------------------------------------------------------------------------

Question: What did Prime Minister Abiy Ahmed pledge to do in 2018?
Expected answer: Reform the country
Answer given: He pledged to reform the country.
ID: ethiopia_challenges__btithihhtt
Score: 5

--------------------------------------------------------------------------------

Question: What group did Abiy Ahmed begin peace talks with in 2018?
Expected answer: Tigray People's Liberation Front
Answer given: The Tigray People’s Liberation Front (TPLF).
ID: ethiopia_challenges__btithihhtt
Score: 5

--------------------------------------------------------------------------------

Question: What percentage of female characters in top 2017 films had speaking roles?
Expected 